# Kaggle Training V4.0 — DermaFusion 24-Class
**Dataset:** ISIC2019 + DermNet + PAD-UFES-20 → 24 classes  
**Model:** EfficientNet-B4 (ImageNet pretrained → fine-tune all layers)

**Outputs:**
- `efficientnet_b4_derma_v4_0.pth` — best checkpoint
- `training_history_v4_0.json` — full per-epoch log
- `fig1_learning_curves.png` — loss / acc / F1 / LR
- `fig2_confusion_matrix.png` — raw + normalized
- `fig3_per_class_metrics.png` — per-class precision / recall / F1
- `fig4_roc_curves.png` — OvR ROC curves (24 classes)
- `fig5_precision_recall_curves.png` — OvR PR curves
- `fig6_class_distribution.png` — train/val sample counts
- `fig7_per_class_f1_trend.png` — F1 trajectory for hardest classes


In [ ]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os, glob, random, json, warnings, itertools
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from tqdm.auto import tqdm
import torch.nn.functional as F
from sklearn.metrics import (f1_score, classification_report, confusion_matrix,
                              roc_curve, auc, precision_recall_curve,
                              average_precision_score)
from sklearn.preprocessing import label_binarize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings('ignore')
print('Imports OK')


In [ ]:
# ============================================================
# CELL 2: SETTINGS & REPRODUCIBILITY
# ============================================================
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

set_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cudnn.benchmark    = True
    torch.backends.cudnn.deterministic = False
print(f'PyTorch: {torch.__version__}')


In [ ]:
# ============================================================
# CELL 3: DATASET DISCOVERY
# ============================================================
EXTRACT_DIR = None

KNOWN_PATHS = [
    '/kaggle/input/derma-db-24class/processed',
    '/kaggle/input/dermadb-24class-v2/processed',
    '/kaggle/input/dermafusion-processed/processed',
    '/kaggle/input/dermafusion-24class/processed',
    '/kaggle/input/datasets/dnghongkhang/derma-db-24class/processed',
    '/kaggle/input/datasets/khang2222/derma-db-24class/processed',
]

for p in KNOWN_PATHS:
    if os.path.exists(os.path.join(p, 'train')):
        EXTRACT_DIR = p
        print(f'✅ Found dataset (known path): {EXTRACT_DIR}')
        break

if not EXTRACT_DIR:
    for root, dirs, _ in itertools.islice(os.walk('/kaggle/input'), 400):
        if 'train' in dirs and 'val' in dirs:
            EXTRACT_DIR = root
            print(f'✅ Found dataset (walk): {EXTRACT_DIR}')
            break

if not EXTRACT_DIR:
    raise RuntimeError('❌ Dataset NOT FOUND — add derma-db-24class dataset.')

# class_weights.json (generated by prepare_dataset_local.py)
CW_SEARCH = [
    os.path.join(os.path.dirname(EXTRACT_DIR), 'class_weights.json'),
    '/kaggle/input/derma-db-24class/class_weights.json',
    '/kaggle/input/dermadb-24class-v2/class_weights.json',
    '/kaggle/input/dermafusion-processed/class_weights.json',
    '/kaggle/input/datasets/khang2222/derma-db-24class/class_weights.json',
]
CLASS_WEIGHTS_PATH = next((p for p in CW_SEARCH if os.path.exists(p)), None)
if CLASS_WEIGHTS_PATH:
    print(f'✅ class_weights.json: {CLASS_WEIGHTS_PATH}')
else:
    print('⚠️  class_weights.json not found — will compute from dataset (fallback)')


In [ ]:
# ============================================================
# CELL 4: AUGMENTATIONS
# ============================================================
IMG_SIZE      = 380          # matches HybridPreprocessingPipeline target_size
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=5)], p=0.2),
    transforms.RandomApply([transforms.RandomAdjustSharpness(sharpness_factor=2)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
])

val_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 20),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# TTA: 3 deterministic transforms, average softmax
tta_tfms_list = [
    transforms.Compose([transforms.Resize(IMG_SIZE + 20), transforms.CenterCrop(IMG_SIZE),
                         transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    transforms.Compose([transforms.Resize(IMG_SIZE + 20), transforms.CenterCrop(IMG_SIZE),
                         transforms.RandomHorizontalFlip(p=1.0),
                         transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    transforms.Compose([transforms.Resize(IMG_SIZE + 20), transforms.CenterCrop(IMG_SIZE),
                         transforms.RandomVerticalFlip(p=1.0),
                         transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
]
print('Augmentations OK')


In [ ]:
# ============================================================
# CELL 5: DATA LOADING & CLASS WEIGHTS
# ============================================================
BATCH_SIZE  = 48
ACCUM_STEPS = 1    # increase to 2 if OOM → effective batch = 96
NUM_WORKERS = 4

train_dir = os.path.join(EXTRACT_DIR, 'train')
val_dir   = os.path.join(EXTRACT_DIR, 'val')

train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds   = datasets.ImageFolder(val_dir,   transform=val_tfms)

class_names = train_ds.classes
NUM_CLASSES = len(class_names)
print(f'Classes ({NUM_CLASSES}): {class_names}')

# ── Class weights ──
# Priority: load from JSON (matches prepare_dataset_local.py formula)
# Fallback: compute balanced weights from actual distribution
if CLASS_WEIGHTS_PATH:
    with open(CLASS_WEIGHTS_PATH) as f:
        cw_dict = json.load(f)
    class_weights_arr = np.array([cw_dict.get(cn, 1.0) for cn in class_names], dtype=np.float32)
    print(f'✅ Loaded class_weights.json')
else:
    labels_all = [y for _, y in train_ds.samples]
    counts     = np.bincount(labels_all, minlength=NUM_CLASSES).astype(np.float32)
    total      = counts.sum()
    class_weights_arr = total / (NUM_CLASSES * (counts + 1e-6))
    print('⚠️  Fallback: computed class weights from dataset distribution')

# Clip to avoid extreme gradient magnitudes
class_weights_arr    = np.clip(class_weights_arr, 0.5, 10.0)
class_weights_tensor = torch.tensor(class_weights_arr, dtype=torch.float32, device=device)

print('\nClass weights (desc):')
for cn, w in sorted(zip(class_names, class_weights_arr), key=lambda x: -x[1]):
    print(f'  {cn:35s}: {w:.4f}')

# ── WeightedRandomSampler ──
labels_all     = [y for _, y in train_ds.samples]
sample_weights = [float(class_weights_arr[lbl]) for lbl in labels_all]
sampler = WeightedRandomSampler(weights=sample_weights,
                                num_samples=len(sample_weights),
                                replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          prefetch_factor=2 if NUM_WORKERS > 0 else None,
                          persistent_workers=(NUM_WORKERS > 0),
                          worker_init_fn=worker_init_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          prefetch_factor=2 if NUM_WORKERS > 0 else None,
                          persistent_workers=(NUM_WORKERS > 0))

print(f'\nTrain batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Effective batch size: {BATCH_SIZE * ACCUM_STEPS}')

# Store counts for fig6
actual_counts_train = np.bincount(labels_all, minlength=NUM_CLASSES)
val_labels_all      = [y for _, y in val_ds.samples]
actual_counts_val   = np.bincount(val_labels_all, minlength=NUM_CLASSES)

print('\nClass distribution (train):')
for cn, cnt in sorted(zip(class_names, actual_counts_train), key=lambda x: x[1]):
    print(f'  {cn:35s}: {cnt:5d}  {"█" * (cnt // 100)}')


## Fig 6 — Class Distribution
Generated from dataset before training starts.

In [ ]:
# ============================================================
# FIG 6: CLASS DISTRIBUTION (train + val)
# ============================================================
sort_idx  = np.argsort(actual_counts_train)[::-1]
cn_sorted = [class_names[i] for i in sort_idx]
tr_sorted = actual_counts_train[sort_idx]
va_sorted = actual_counts_val[sort_idx]

x = np.arange(NUM_CLASSES)
w = 0.4

fig, axes = plt.subplots(2, 1, figsize=(18, 14))

# ── Top: stacked bar ──
axes[0].bar(x, tr_sorted, width=0.6, color='steelblue', label='Train', alpha=0.85)
axes[0].bar(x, va_sorted, width=0.6, bottom=tr_sorted, color='coral', label='Val', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(cn_sorted, rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Sample count')
axes[0].set_title('Class Distribution — Train + Val (stacked)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for i, (t, v) in enumerate(zip(tr_sorted, va_sorted)):
    axes[0].text(i, t + v + 30, str(t + v), ha='center', va='bottom', fontsize=7, rotation=90)

# ── Bottom: class weight bar ──
cw_sorted = class_weights_arr[sort_idx]
bar_colors = ['#d62728' if w > 3 else '#ff7f0e' if w > 1.5 else '#2ca02c' for w in cw_sorted]
axes[1].bar(x, cw_sorted, color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].axhline(1.0, color='k', linestyle='--', linewidth=1, label='weight=1 (balanced)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(cn_sorted, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Class weight')
axes[1].set_title('Class Weights (red > 3×, orange 1.5-3×, green ≤1.5×)', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('DermaFusion 24-Class — Dataset Statistics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig6_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig6_class_distribution.png')


## Model Setup

In [ ]:
# ============================================================
# CELL 7: MODEL — EfficientNet-B4
# ============================================================
model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)

in_features = model.classifier[1].in_features

# 2-layer head: BN + SiLU matches EfficientNet's internal activations
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=False),
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.SiLU(),
    nn.Dropout(p=0.3, inplace=False),
    nn.Linear(512, NUM_CLASSES)
)

model = model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model    : EfficientNet-B4 (ImageNet pretrained)')
print(f'Head     : {in_features} → 512 → {NUM_CLASSES}')
print(f'Params   : {total/1e6:.2f}M total | {trainable/1e6:.2f}M trainable')


In [ ]:
# ============================================================
# CELL 8: LOSS — Asymmetric + Label Smoothing + Class Weighting
# ============================================================
class AsymmetricLoss(nn.Module):
    """
    Asymmetric Focal Loss with label smoothing and per-sample class weighting.

    gamma_neg > gamma_pos  → penalise false negatives harder (missing cancer is worse)
    smoothing              → prevent overconfident predictions, improves calibration
    class_weights          → per-sample gradient scaling for imbalanced classes
    """
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05,
                 smoothing=0.1, class_weights=None):
        super().__init__()
        self.gamma_neg  = gamma_neg
        self.gamma_pos  = gamma_pos
        self.clip       = clip
        self.smoothing  = smoothing
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.class_weights = None

    def forward(self, logits, targets):
        C = logits.size(1)
        # Label smoothing
        y = F.one_hot(targets, num_classes=C).float()
        if self.smoothing > 0:
            y = y * (1 - self.smoothing) + self.smoothing / C

        p     = torch.softmax(logits, dim=1)
        p_neg = (1 - p + self.clip).clamp(max=1)

        loss = (
            -y         * (1 - p)    ** self.gamma_pos * torch.log(p.clamp(1e-8))
            - (1 - y)  * p          ** self.gamma_neg * torch.log(p_neg.clamp(1e-8))
        )  # (B, C)

        loss_per_sample = loss.sum(dim=1)  # (B,)

        if self.class_weights is not None:
            return (loss_per_sample * self.class_weights[targets]).mean()
        return loss_per_sample.mean()


criterion = AsymmetricLoss(
    gamma_neg=4.0, gamma_pos=1.0, smoothing=0.1,
    class_weights=class_weights_tensor
).to(device)

print('Loss: AsymmetricLoss(gamma_neg=4, gamma_pos=1, smoothing=0.1, class_weighted=True)')


In [ ]:
# ============================================================
# CELL 9: OPTIMIZER & SCHEDULER
# ============================================================
BASE_LR_FEATURES = 5e-5
BASE_LR_HEAD     = 3e-4
WARMUP_EPOCHS    = 5
NUM_EPOCHS       = 40
WEIGHT_DECAY     = 1e-3

optimizer = torch.optim.AdamW([
    {'params': model.features.parameters(),   'lr': BASE_LR_FEATURES},
    {'params': model.classifier.parameters(), 'lr': BASE_LR_HEAD},
], weight_decay=WEIGHT_DECAY)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=1e-7
)

def apply_warmup(epoch_0idx, warmup_epochs, base_lrs):
    """Linear warmup. epoch_0idx = epoch - 1 (0-indexed)."""
    if epoch_0idx < warmup_epochs:
        scale = (epoch_0idx + 1) / warmup_epochs
        for i, pg in enumerate(optimizer.param_groups):
            pg['lr'] = base_lrs[i] * scale
        return True
    return False

base_lrs = [BASE_LR_FEATURES, BASE_LR_HEAD]
scaler   = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

print(f'Optimizer : AdamW | features lr={BASE_LR_FEATURES} | head lr={BASE_LR_HEAD} | wd={WEIGHT_DECAY}')
print(f'Scheduler : Linear warmup ({WARMUP_EPOCHS} ep) → CosineAnnealing ({NUM_EPOCHS - WARMUP_EPOCHS} ep)')
print(f'Epochs    : {NUM_EPOCHS} | ACCUM_STEPS: {ACCUM_STEPS}')


## Training Loop

In [ ]:
# ============================================================
# CELL 10: TRAINING LOOP
# ============================================================
OUT_NAME  = 'efficientnet_b4_derma_v4_0.pth'
HIST_FILE = 'training_history_v4_0.json'
PATIENCE  = 12

best_val_f1   = 0.0
epochs_no_imp = 0
history       = []

print(f'Starting V4.0 training ({NUM_EPOCHS} epochs, warmup={WARMUP_EPOCHS})...')
print(f'Checkpoint → {OUT_NAME}')
print()

for epoch in range(1, NUM_EPOCHS + 1):

    is_warmup  = apply_warmup(epoch - 1, WARMUP_EPOCHS, base_lrs)
    current_lr = optimizer.param_groups[0]['lr']

    # ── TRAIN ──────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (images, labels_b) in enumerate(
            tqdm(train_loader, desc=f'Ep{epoch:02d} train', leave=False)):
        images   = images.to(device, non_blocking=True)
        labels_b = labels_b.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss   = criterion(logits, labels_b) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        train_loss += loss.item() * ACCUM_STEPS * images.size(0)

    train_loss /= len(train_loader.dataset)
    if not is_warmup:
        cosine_scheduler.step()

    # ── VALIDATE ───────────────────────────────────────────
    model.eval()
    val_loss   = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels_b in tqdm(val_loader, desc=f'Ep{epoch:02d} val  ', leave=False):
            images   = images.to(device, non_blocking=True)
            labels_b = labels_b.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(images)
                loss   = criterion(logits, labels_b)
            val_loss   += loss.item() * images.size(0)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels_b.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    val_f1    = f1_score(all_labels, all_preds, average='macro',  zero_division=0)
    val_acc   = (np.array(all_preds) == np.array(all_labels)).mean()
    pcf1      = f1_score(all_labels, all_preds, average=None, zero_division=0)

    worst3_str = ', '.join(
        f'{cn}={v:.2f}' for cn, v in
        sorted(zip(class_names, pcf1), key=lambda x: x[1])[:3]
    )

    phase = 'WARMUP' if is_warmup else 'COSINE'
    history.append({
        'epoch': epoch, 'phase': phase,
        'lr': round(current_lr, 8),
        'train_loss': round(train_loss, 4),
        'val_loss':   round(val_loss,   4),
        'val_acc':    round(float(val_acc),  4),
        'val_f1':     round(float(val_f1),   4),
        'per_class_f1': {cn: round(float(v), 4) for cn, v in zip(class_names, pcf1)},
    })

    print(f'Ep {epoch:2d}/{NUM_EPOCHS} [{phase}] lr={current_lr:.2e} | '
          f'train={train_loss:.4f} | val={val_loss:.4f} | '
          f'acc={val_acc:.4f} | f1={val_f1:.4f} | worst:[{worst3_str}]')

    if val_f1 > best_val_f1:
        best_val_f1   = val_f1
        epochs_no_imp = 0
        tmp = OUT_NAME + '.tmp'
        torch.save({
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'epoch': epoch,
            'val_f1':      best_val_f1,
            'val_acc':     float(val_acc),
            'class_names': class_names,
            'num_classes': NUM_CLASSES,
            'img_size':    IMG_SIZE,
            'per_class_f1': {cn: round(float(v), 4) for cn, v in zip(class_names, pcf1)},
        }, tmp)
        os.replace(tmp, OUT_NAME)
        print(f'  ✓ Checkpoint saved (F1={best_val_f1:.4f}, Acc={val_acc:.4f})')
    else:
        epochs_no_imp += 1

    if epochs_no_imp >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

# Restore best
best_ckpt = torch.load(OUT_NAME, map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state'])
print(f'\n✅ Best model restored — epoch {best_ckpt["epoch"]} | '
      f'F1={best_ckpt["val_f1"]:.4f} | Acc={best_ckpt["val_acc"]:.4f}')

with open(HIST_FILE, 'w') as f:
    json.dump(history, f, indent=2)
print(f'History saved → {HIST_FILE}')


## Figures — generated after training

In [ ]:
# ============================================================
# FIG 1: LEARNING CURVES
# ============================================================
with open(HIST_FILE) as f:
    h = json.load(f)

ep   = [e['epoch']      for e in h]
tl   = [e['train_loss'] for e in h]
vl   = [e['val_loss']   for e in h]
vacc = [e['val_acc']    for e in h]
vf1  = [e['val_f1']     for e in h]
lrs  = [e['lr']         for e in h]
wu   = [e['epoch'] for e in h if e['phase'] == 'WARMUP']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('V4.0 Training Curves — DermaFusion 24-class', fontsize=14, fontweight='bold')

def shade_warmup(ax):
    if wu:
        ax.axvspan(wu[0] - 0.5, wu[-1] + 0.5, alpha=0.08, color='orange', label='Warmup')

# Loss
ax = axes[0, 0]
ax.plot(ep, tl, 'b-', lw=1.5, label='Train loss')
ax.plot(ep, vl, 'r-', lw=1.5, label='Val loss')
shade_warmup(ax)
ax.set_title('Loss'); ax.legend(); ax.grid(alpha=0.3)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')

# Accuracy
ax = axes[0, 1]
ax.plot(ep, vacc, 'g-', lw=2)
ax.axhline(max(vacc), color='g', ls='--', alpha=0.5, label=f'Best {max(vacc):.4f}')
shade_warmup(ax)
ax.set_title('Val Accuracy'); ax.legend(); ax.set_ylim([0, 1]); ax.grid(alpha=0.3)
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')

# F1
ax = axes[1, 0]
ax.plot(ep, vf1, 'm-', lw=2)
ax.axhline(max(vf1), color='m', ls='--', alpha=0.5, label=f'Best {max(vf1):.4f}')
shade_warmup(ax)
ax.set_title('Val F1 (Macro)'); ax.legend(); ax.set_ylim([0, 1]); ax.grid(alpha=0.3)
ax.set_xlabel('Epoch'); ax.set_ylabel('F1 Macro')

# LR
ax = axes[1, 1]
ax.plot(ep, lrs, 'c-', lw=2)
ax.set_yscale('log'); ax.set_title('Learning Rate')
ax.grid(alpha=0.3); ax.set_xlabel('Epoch'); ax.set_ylabel('LR (log scale)')

plt.tight_layout()
plt.savefig('fig1_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig1_learning_curves.png')


In [ ]:
# ============================================================
# TTA EVALUATION — collect probs for all downstream figures
# ============================================================
print('Running TTA evaluation (3 transforms)...')
model.eval()

tta_probs_sum = None
true_labels   = None

for t_idx, tta_tfm in enumerate(tta_tfms_list):
    tta_ds  = datasets.ImageFolder(val_dir, transform=tta_tfm)
    tta_ldr = DataLoader(tta_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

    probs_list = []
    with torch.no_grad():
        for imgs, _ in tqdm(tta_ldr, desc=f'TTA {t_idx+1}/3', leave=False):
            imgs = imgs.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type=='cuda')):
                logits = model(imgs)
            probs_list.append(torch.softmax(logits.float(), dim=1).cpu())

    probs_t = torch.cat(probs_list, dim=0)  # (N, C)
    tta_probs_sum = probs_t if tta_probs_sum is None else tta_probs_sum + probs_t

    if true_labels is None:
        true_labels = np.array([y for _, y in tta_ds.samples])

tta_probs = (tta_probs_sum / len(tta_tfms_list)).numpy()  # (N, C) float
tta_preds = tta_probs.argmax(axis=1)

tta_f1  = f1_score(true_labels, tta_preds, average='macro', zero_division=0)
tta_acc = (tta_preds == true_labels).mean()

print(f'\nTTA Results — Acc: {tta_acc:.4f} | F1 macro: {tta_f1:.4f}')
print(f'(No-TTA best — Acc: {best_ckpt["val_acc"]:.4f} | F1: {best_ckpt["val_f1"]:.4f})')

# Binarise labels for ROC / PR
labels_bin = label_binarize(true_labels, classes=list(range(NUM_CLASSES)))


In [ ]:
# ============================================================
# FIG 2: CONFUSION MATRIX (raw + normalised)
# ============================================================
cm      = confusion_matrix(true_labels, tta_preds)
cm_norm = np.nan_to_num(cm.astype(float) / cm.sum(axis=1, keepdims=True))

fig, axes = plt.subplots(1, 2, figsize=(38, 15))

kw = dict(xticklabels=class_names, yticklabels=class_names)
sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues', ax=axes[0], **kw)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
            vmin=0, vmax=1, **kw)

for ax, title in zip(axes, ['Raw Counts', 'Normalised (per row)']):
    ax.set_title(f'Confusion Matrix — {title}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    plt.setp(ax.get_yticklabels(), rotation=0,  fontsize=8)

fig.suptitle(f'V4.0 Confusion Matrix (TTA) — Acc={tta_acc:.4f} | F1={tta_f1:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig2_confusion_matrix.png')


In [ ]:
# ============================================================
# FIG 3: PER-CLASS METRICS (Precision / Recall / F1)
# ============================================================
from sklearn.metrics import precision_score, recall_score

prec_per = precision_score(true_labels, tta_preds, average=None, zero_division=0)
rec_per  = recall_score(   true_labels, tta_preds, average=None, zero_division=0)
f1_per   = f1_score(       true_labels, tta_preds, average=None, zero_division=0)
support  = np.bincount(true_labels, minlength=NUM_CLASSES)

# Sort by F1 descending
sort_idx  = np.argsort(f1_per)[::-1]
cn_s      = [class_names[i] for i in sort_idx]
f1_s      = f1_per[sort_idx]
prec_s    = prec_per[sort_idx]
rec_s     = rec_per[sort_idx]
sup_s     = support[sort_idx]

x = np.arange(NUM_CLASSES)
w = 0.26

fig, axes = plt.subplots(2, 1, figsize=(20, 14))

# ── Top: grouped bar ──
axes[0].bar(x - w, prec_s, width=w, label='Precision', color='steelblue',  alpha=0.85)
axes[0].bar(x,     f1_s,   width=w, label='F1',        color='mediumseagreen', alpha=0.85)
axes[0].bar(x + w, rec_s,  width=w, label='Recall',    color='coral',      alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(cn_s, rotation=45, ha='right', fontsize=9)
axes[0].set_ylim([0, 1.05]); axes[0].set_ylabel('Score')
axes[0].set_title('Per-Class Precision / F1 / Recall (sorted by F1 desc)',
                  fontsize=12, fontweight='bold')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
axes[0].axhline(tta_f1, color='purple', ls='--', lw=1.2, label=f'Macro F1={tta_f1:.3f}')

# Value labels on F1 bars
for i, v in enumerate(f1_s):
    axes[0].text(i, v + 0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=7)

# ── Bottom: support (sample count per class) ──
colors = ['#d62728' if s < 100 else '#ff7f0e' if s < 300 else '#2ca02c' for s in sup_s]
axes[1].bar(x, sup_s, color=colors, alpha=0.85, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(cn_s, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Val samples'); axes[1].set_yscale('log')
axes[1].set_title('Val Set Support per Class (log scale — red<100, orange<300, green≥300)',
                  fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for i, s in enumerate(sup_s):
    axes[1].text(i, s * 1.05, str(s), ha='center', va='bottom', fontsize=7, rotation=90)

fig.suptitle('V4.0 Per-Class Metrics (TTA)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig3_per_class_metrics.png')


In [ ]:
# ============================================================
# FIG 4: ROC CURVES (One-vs-Rest, all 24 classes)
# ============================================================
fpr_dict, tpr_dict, roc_auc_dict = {}, {}, {}

for i in range(NUM_CLASSES):
    fpr_dict[i], tpr_dict[i], _ = roc_curve(labels_bin[:, i], tta_probs[:, i])
    roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

# Macro-average AUC
mean_auc = np.mean(list(roc_auc_dict.values()))

# Sort classes by AUC descending for the legend
sorted_by_auc = sorted(range(NUM_CLASSES), key=lambda i: -roc_auc_dict[i])

# Use a 4×6 grid of subplots (one per class)
fig, axes = plt.subplots(4, 6, figsize=(26, 18))
axes_flat = axes.flatten()

cmap = plt.cm.get_cmap('tab20', NUM_CLASSES)

for plot_idx, class_idx in enumerate(sorted_by_auc):
    ax = axes_flat[plot_idx]
    ax.plot(fpr_dict[class_idx], tpr_dict[class_idx],
            color=cmap(plot_idx), lw=1.8,
            label=f'AUC={roc_auc_dict[class_idx]:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_title(class_names[class_idx].replace('_', '\n'), fontsize=7, fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')
    ax.set_xticks([]); ax.set_yticks([0, 0.5, 1])
    ax.grid(alpha=0.2)

# Remove any unused subplots
for i in range(NUM_CLASSES, len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'V4.0 ROC Curves — One-vs-Rest (TTA)  |  Macro AUC = {mean_auc:.4f}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved fig4_roc_curves.png  |  Macro AUC = {mean_auc:.4f}')


In [ ]:
# ============================================================
# FIG 5: PRECISION-RECALL CURVES (One-vs-Rest, all 24 classes)
# ============================================================
pr_prec, pr_rec, pr_ap = {}, {}, {}

for i in range(NUM_CLASSES):
    pr_prec[i], pr_rec[i], _ = precision_recall_curve(labels_bin[:, i], tta_probs[:, i])
    pr_ap[i] = average_precision_score(labels_bin[:, i], tta_probs[:, i])

mean_ap = np.mean(list(pr_ap.values()))
sorted_by_ap = sorted(range(NUM_CLASSES), key=lambda i: -pr_ap[i])

# Baseline (random) per class
baselines = support / support.sum()  # = class prevalence in val set

fig, axes = plt.subplots(4, 6, figsize=(26, 18))
axes_flat  = axes.flatten()

for plot_idx, class_idx in enumerate(sorted_by_ap):
    ax = axes_flat[plot_idx]
    ax.plot(pr_rec[class_idx], pr_prec[class_idx],
            color=cmap(plot_idx), lw=1.8,
            label=f'AP={pr_ap[class_idx]:.3f}')
    ax.axhline(baselines[class_idx], color='k', ls='--', lw=0.8, alpha=0.5, label='chance')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    ax.set_title(class_names[class_idx].replace('_', '\n'), fontsize=7, fontweight='bold')
    ax.legend(fontsize=7, loc='upper right')
    ax.set_xticks([]); ax.set_yticks([0, 0.5, 1])
    ax.grid(alpha=0.2)

for i in range(NUM_CLASSES, len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'V4.0 Precision-Recall Curves — One-vs-Rest (TTA)  |  mAP = {mean_ap:.4f}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved fig5_precision_recall_curves.png  |  mAP = {mean_ap:.4f}')


In [ ]:
# ============================================================
# FIG 7: PER-CLASS F1 TREND (8 hardest classes over epochs)
# ============================================================
with open(HIST_FILE) as f:
    h = json.load(f)

pcf1_history = [e for e in h if 'per_class_f1' in e]

if pcf1_history:
    last_pcf1 = pcf1_history[-1]['per_class_f1']
    worst8    = [cn for cn, _ in sorted(last_pcf1.items(), key=lambda x: x[1])[:8]]
    best8     = [cn for cn, _ in sorted(last_pcf1.items(), key=lambda x: -x[1])[:4]]

    fig, axes = plt.subplots(1, 2, figsize=(20, 7))

    # Hardest 8
    for cn in worst8:
        trend = [e['per_class_f1'].get(cn, 0) for e in pcf1_history]
        eps   = [e['epoch'] for e in pcf1_history]
        axes[0].plot(eps, trend, marker='o', ms=3, lw=1.5, label=cn)
    axes[0].set_title('8 Hardest Classes — F1 over Epochs', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('F1')
    axes[0].set_ylim([0, 1]); axes[0].legend(fontsize=8, loc='upper left')
    axes[0].grid(alpha=0.3)

    # Best 4 (sanity check — should be high and stable)
    for cn in best8:
        trend = [e['per_class_f1'].get(cn, 0) for e in pcf1_history]
        eps   = [e['epoch'] for e in pcf1_history]
        axes[1].plot(eps, trend, marker='s', ms=3, lw=1.5, label=cn)
    axes[1].set_title('4 Best Classes — F1 over Epochs', fontsize=11, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1')
    axes[1].set_ylim([0, 1]); axes[1].legend(fontsize=8, loc='lower right')
    axes[1].grid(alpha=0.3)

    fig.suptitle('V4.0 Per-Class F1 Trajectory', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig7_per_class_f1_trend.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fig7_per_class_f1_trend.png')
else:
    print('No per_class_f1 in history.')


In [ ]:
# ============================================================
# CLASSIFICATION REPORT & FINAL SUMMARY
# ============================================================
report = classification_report(true_labels, tta_preds,
                                target_names=class_names, zero_division=0)
print(report)

with open('v4_0_classification_report.txt', 'w') as f:
    f.write('V4.0 Classification Report (TTA)\n')
    f.write(f'Acc={tta_acc:.4f}  F1_macro={tta_f1:.4f}  mAP={mean_ap:.4f}  AUC={mean_auc:.4f}\n\n')
    f.write(report)

print('\n' + '='*60)
print('V4.0  FINAL SUMMARY')
print('='*60)
rows = [
    ('Dataset',      'ISIC2019 + DermNet + PAD-UFES-20 (24 classes)'),
    ('Model',        'EfficientNet-B4 (ImageNet pretrained)'),
    ('IMG_SIZE',     f'{IMG_SIZE}×{IMG_SIZE}'),
    ('Batch size',   f'{BATCH_SIZE} (eff: {BATCH_SIZE*ACCUM_STEPS})'),
    ('Epochs ran',   f'{len(history)}/{NUM_EPOCHS}'),
    ('Best epoch',   str(best_ckpt["epoch"])),
    ('Val F1',       f'{best_ckpt["val_f1"]:.4f}  (no TTA)'),
    ('Val Acc',      f'{best_ckpt["val_acc"]:.4f}  (no TTA)'),
    ('TTA F1',       f'{tta_f1:.4f}'),
    ('TTA Acc',      f'{tta_acc:.4f}'),
    ('Macro AUC',    f'{mean_auc:.4f}'),
    ('mAP',          f'{mean_ap:.4f}'),
]
for k, v in rows:
    print(f'  {k:14s}: {v}')

print('\nOutput files:')
outputs = [
    'efficientnet_b4_derma_v4_0.pth',
    'training_history_v4_0.json',
    'fig1_learning_curves.png',
    'fig2_confusion_matrix.png',
    'fig3_per_class_metrics.png',
    'fig4_roc_curves.png',
    'fig5_precision_recall_curves.png',
    'fig6_class_distribution.png',
    'fig7_per_class_f1_trend.png',
    'v4_0_classification_report.txt',
]
for fname in outputs:
    mark = '✅' if os.path.exists(fname) else '❌'
    size = f'({os.path.getsize(fname)/1e6:.1f} MB)' if os.path.exists(fname) else ''
    print(f'  {mark} {fname} {size}')
print('='*60)
